# Week 2 – Day 2: Decision Trees & Ensemble Methods (Random Forest)

## Objective & Curriculum Scope
This notebook explores the foundations of non-linear decision boundaries using tree-based architectures on the standard **Wine recognition dataset**:
1. **Decision Tree Classifier:** Splitting criteria (Gini vs. Shannon Entropy), hyperparameter optimization via 5-fold cross-validation (`max_depth`, `min_samples_leaf`), visual inspection of tree decision rules, and feature importance derivation.
2. **Theory & Mechanics:** Root split analysis, intuitive interpretation of Gini impurity, and variance reduction via Bootstrap Aggregation (Bagging).
3. **Random Forest Ensembles:** Out-of-bag (OOB) error estimation, tree-depth stabilization, impact of ensemble size (`n_estimators`), and feature attribution comparison.
4. **Ensemble Variants (Bonus):** Evaluation of Extremely Randomized Trees (`ExtraTreesClassifier`).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report

# Configure plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Load the multiclass Wine recognition dataset
wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names
target_names = wine.target_names

print(f"Dataset Dimensions: {X.shape[0]} samples, {X.shape[1]} chemical features")
print(f"Target Classes: {list(target_names)}")

# Stratified 80/20 train/test split to preserve wine cultivar distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train samples: {X_train.shape[0]} | Held-out test samples: {X_test.shape[0]}")

In [ ]:
# --- TASK 1: SPLITTING CRITERIA COMPARISON ---
print("--- 1. Evaluating Splitting Criteria ---")
for criterion in ['gini', 'entropy', 'log_loss']:
    dt_clf = DecisionTreeClassifier(criterion=criterion, random_state=42).fit(X_train, y_train)
    acc = accuracy_score(y_test, dt_clf.predict(X_test))
    print(f"Decision Tree (Criterion='{criterion:<8}'): Test Accuracy = {acc:.4f}")

# --- TASK 1: HYPERPARAMETER TUNING VIA GRID SEARCH ---
param_grid = {
    'max_depth': [2, 3, 4, 5, None],
    'min_samples_leaf': [1, 2, 4, 6]
}

dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
tuned_acc = accuracy_score(y_test, best_dt.predict(X_test))
print(f"\nOptimal Hyperparameters: {dt_grid.best_params_}")
print(f"Tuned Decision Tree Test Accuracy: {tuned_acc:.4f}")

In [ ]:
# --- TASK 1: TREE STRUCTURE VISUALIZATION ---
dt_visual = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

plt.figure(figsize=(18, 9), dpi=300)
plot_tree(
    dt_visual,
    feature_names=feature_names,
    class_names=target_names,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Trained Decision Tree (max_depth=3) - Decision Logic Flowchart", fontsize=14, fontweight='bold', pad=12)
plt.savefig("wine_tree.png", bbox_inches='tight', dpi=300)
plt.show()

# Top Feature Importances from the Decision Tree
dt_importances = pd.Series(dt_visual.feature_importances_, index=feature_names).sort_values(ascending=False)
print("Top 5 Informative Features (Decision Tree):")
print(dt_importances.head(5).to_markdown())

### Task 1: Mechanistic Analysis of the Decision Tree

#### 1. Splitting Logic (Root & Secondary Nodes)
* **Root Split (`proline <= 755.0`):** The tree identifies proline concentration as the single most discriminative feature for separating class 0 (cultivar 0) from cultivars 1 and 2. Wines with `proline > 755.0` route directly to the right branch and achieve nearly pure classification as class 0.
* **Secondary Split (`od280/od315_of_diluted_wines <= 2.115`):** For the remaining sub-population on the left (`proline <= 755.0`), the algorithm evaluates the protein dilution ratio. Values `<= 2.115` indicate cultivar 2, whereas higher values proceed toward cultivar 1, effectively partitioning the feature space along orthogonal hyperplanes.

#### 2. Intuitive Definition of Gini Impurity (3 Lines)
1. **Purity Metric:** Gini impurity measures the statistical probability that a randomly drawn sample from a node would be incorrectly labeled if assigned a random class according to the node's class distribution.
2. **Boundary Values:** A score of `0.0` represents complete purity (every instance belongs to the exact same class), whereas a maximum score (~0.67 for a 3-class problem) denotes a uniform mixture of classes.
3. **Split Objective:** At each node, the decision tree algorithm tests every candidate feature threshold and selects the split that maximizes the reduction in weighted Gini impurity (producing the cleanest child subsets).

#### 3. Bootstrap Aggregation (Bagging) & Overfitting Reduction
* **Single Tree Vulnerability:** A single unconstrained decision tree creates high-variance partitions that memorize specific noise in the training observations, leading to severe overfitting and poor test generalization.
* **Bagging Mechanism:** Random Forests construct an ensemble of independent trees (e.g., 200 estimators). Each tree is trained on a bootstrap sample (drawn uniformly at random with replacement from the training rows) and considers only a random subset of features at each candidate split (`mtry = sqrt(p)`).
* **Variance Cancellation:** Because individual trees are decorrelated through randomized row and feature sampling, their idiosyncratic errors and high-variance spikes average out during majority voting, substantially reducing overall generalization error without inflating model bias.

In [ ]:
# --- TASK 2: RANDOM FOREST WITH OUT-OF-BAG (OOB) SCORING ---
rf_model = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
rf_model.fit(X_train, y_train)

single_tree_acc = accuracy_score(y_test, dt_visual.predict(X_test))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test))
rf_oob = rf_model.oob_score_

print(f"Single Decision Tree Test Accuracy : {single_tree_acc:.4f}")
print(f"Random Forest (200 Trees) Test Acc : {rf_test_acc:.4f}")
print(f"Random Forest Out-of-Bag (OOB) Score: {rf_oob:.4f}")
print("\nClassification Report (Random Forest held-out test):\n")
print(classification_report(y_test, rf_model.predict(X_test), target_names=target_names))

In [ ]:
# --- TASK 2: SENSITIVITY TO ENSEMBLE SIZE (n_estimators) ---
n_estimators_list = [10, 50, 100, 200, 300]
rf_accuracies = []

for n in n_estimators_list:
    clf = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
    rf_accuracies.append(accuracy_score(y_test, clf.predict(X_test)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Accuracy vs n_estimators
ax1.plot(n_estimators_list, rf_accuracies, marker='s', color='#2b5c8f', linewidth=2, markersize=7)
ax1.set_xlabel("Number of Estimators (Trees)", fontsize=11, labelpad=8)
ax1.set_ylabel("Held-Out Test Accuracy", fontsize=11, labelpad=8)
ax1.set_title("Random Forest: Generalization Accuracy vs. n_estimators", fontsize=12, fontweight='bold')
ax1.set_ylim(0.90, 1.02)
ax1.grid(True, linestyle='--', alpha=0.6)

# Plot 2: Random Forest Mean Decrease in Impurity (MDI)
rf_importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values()
rf_importances.plot(kind='barh', color='#2ca02c', ax=ax2, edgecolor='black', alpha=0.85)
ax2.set_xlabel("Mean Decrease in Impurity (Gini Importance)", fontsize=11, labelpad=8)
ax2.set_title("Random Forest: Overall Feature Importances", fontsize=12, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# --- BONUS CHALLENGE: EXTREMELY RANDOMIZED TREES (ExtraTreesClassifier) ---
extra_trees = ExtraTreesClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
et_acc = accuracy_score(y_test, extra_trees.predict(X_test))

print(f"Standard Random Forest Test Accuracy : {rf_test_acc:.4f}")
print(f"ExtraTrees Classifier Test Accuracy  : {et_acc:.4f}")

# Architectural note
print("""
ExtraTrees Analysis:
Unlike Random Forest (which searches for the optimal threshold among candidates for each split),
ExtraTrees picks thresholds purely at random for each candidate feature and selects the best of those random thresholds.
This introduces additional variance reduction and computational speedup at the expense of a slight potential increase in bias.
""")